# Notebook 0 — Production source files

**Which `.java` files in each repository are production code?** Everything downstream (the labels in
Notebook 1, the evidence in Notebook 2 and the texts in Notebook 3) uses the answer, so it is decided
once, here, from evidence.

**The rule.** A file is production if it lies under an **approved source root**. Everything else
is excluded. Two roots are assumed, because they are the Java conventions:

- `src/main/java/` (Maven and Gradle)
- `src/java/` (older Ant layouts)

Any other root must be **found** in the repository, and approved explicitly.

**How roots are found.** A Java file's `package` declaration says where its package tree begins:

```
src/checkstyle/com/puppycrawl/tools/checkstyle/Checker.java
               package com.puppycrawl.tools.checkstyle;
→ root = src/checkstyle/
```

This needs no knowledge of the layout, so it finds historical and unusual roots as reliably as
conventional ones.

**Ground truth is git**, not any miner. Each release's tree says which files existed.

**Writes** (to `outputs/`): `production_scope.json` (the approved roots), `production_files_<repo>.csv`
(every `.java` file at every release, with its category) and `production_summary.csv`. Section 4 also
writes `refactored_scope_summary.csv` and `refactored_roots_<repo>.csv`.

Cell 0 downloads the mined inputs from Zenodo. Sections 1–3 then use git only.

### Cell 0 — The mined inputs, from Zenodo
Notebooks 0–3 read what was mined from the five repos: the release grids, Quarry's outputs and a git
clone of each repo. They are published on Zenodo, [10.5281/zenodo.24182007](https://doi.org/10.5281/zenodo.24182007),
as one zip and one git bundle per repo.

For each repo this cell downloads only what is missing from `workspace_data/`:
- the zip, checked against the record's `SHA256SUMS.txt` and unpacked into `workspace_data/`, every file
  checked against the zip's `MANIFEST_<repo>.json`;
- the bundle, checked the same way and restored as the clone in `workspace_data/repos/<repo>/`.

A workspace that already holds the files and clones (for example one that ran Quarry itself) downloads
nothing. The same cell opens Notebooks 0, 1, 2 and 3, so any of them can be run first. The first run
downloads about 2.5 GB.

In [ ]:
import os, json, hashlib, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

# The workspace Quarry uses: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
MINED_RECORD = "24182007"                         # Zenodo record of the mined inputs
MINED_REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
NEEDED = ["versions/rel_hist_{r}.json", "outputs/{r}/intervals_{r}.json", "outputs/{r}/refactorings_{r}.jsonl",
          "outputs/{r}/ledger_history_{r}.jsonl", "outputs/{r}/diffs_{r}.jsonl", "outputs/{r}/links_{r}.csv"]


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def have_mined_files(repo):
    """Every file the notebooks need, in outputs/<repo>/ or outputs/ (both layouts are read)."""
    def there(rel):
        p = WORKSPACE / rel.format(r=repo)
        return p.exists() or (WORKSPACE / "outputs" / p.name).exists()
    return all(there(x) for x in NEEDED)


def download(name, dest, sums):
    """One file of the record, checked against SHA256SUMS.txt before it is kept."""
    tmp = dest.with_name(dest.name + ".part")
    print(f"   downloading {name}")
    urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/{name}?download=1", tmp)
    if sha256_of(tmp) != sums[name]:
        tmp.unlink()
        raise SystemExit(f"{name}: SHA-256 differs from the record's SHA256SUMS.txt; download it again")
    tmp.replace(dest)


def fetch_mined(repos):
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(parents=True, exist_ok=True)
    sums = None
    for repo in repos:
        clone = WORKSPACE / "repos" / repo
        need_files, need_clone = not have_mined_files(repo), not (clone / ".git").exists()
        if not (need_files or need_clone):
            print(f"{repo:<18}mined files and clone already in {WORKSPACE}")
            continue
        if sums is None:
            urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/SHA256SUMS.txt?download=1",
                                       downloads / "SHA256SUMS.txt")
            sums = {n: h for h, n in (l.split() for l in
                    (downloads / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if l.strip())}
        if need_files:
            z = downloads / f"{repo}_mined.zip"
            download(z.name, z, sums)
            with zipfile.ZipFile(z) as zf:
                manifest = json.loads(zf.read(f"MANIFEST_{repo}.json"))
                for f in manifest["files"]:
                    out = WORKSPACE / f["file"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(f["file"]) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 24)
                    if sha256_of(out) != f["sha256"]:
                        raise SystemExit(f"{f['file']}: SHA-256 differs from MANIFEST_{repo}.json")
                (WORKSPACE / "versions" / f"MANIFEST_{repo}.json").write_bytes(zf.read(f"MANIFEST_{repo}.json"))
            z.unlink()
            print(f"{repo:<18}{len(manifest['files'])} mined files unpacked and checked")
        if need_clone:
            b = downloads / f"{repo}.bundle"
            download(b.name, b, sums)
            clone.mkdir(parents=True, exist_ok=True)
            for args in (["clone", "--mirror", "--quiet", str(b.resolve()), str(clone / ".git")],
                         ["-C", str(clone), "config", "--bool", "core.bare", "false"],
                         ["-C", str(clone), "-c", "core.longpaths=true", "reset", "--hard", "--quiet"]):
                r = subprocess.run(["git", *args], capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(f"git {' '.join(args[:2])} failed for {repo}: {r.stderr.strip()[:300]}")
            b.unlink()
            print(f"{repo:<18}clone restored from {b.name}")


fetch_mined(MINED_REPOS)

In [13]:
import os, re, json, subprocess, collections
from pathlib import Path
import pandas as pd

# The workspace Quarry uses: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
# This notebook lives in <codebase>/notebooks, so the default is one level up.
WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
REPOS_DIR = WORKSPACE / "repos"
VERSIONS = WORKSPACE / "versions"
OUTPUTS = WORKSPACE / "outputs"

# Every repository with a release manifest. Set REPOS by hand to run a subset.
REPOS = sorted(p.stem[len("rel_hist_"):] for p in VERSIONS.glob("rel_hist_*.json"))
REPOS = [r for r in REPOS if (REPOS_DIR / r / ".git").exists()]

# Files that hold no type are never production files.
NO_TYPE = {"package-info.java", "module-info.java"}

print(f"workspace   {WORKSPACE}")
print(f"repositories {', '.join(REPOS) or 'none found'}")
if not REPOS:
    raise SystemExit(f"no rel_hist_<repo>.json in {VERSIONS} with a clone in {REPOS_DIR}")


def git(repo, *args, stdin=None):
    r = subprocess.run(["git", *args], cwd=REPOS_DIR / repo, input=stdin,
                       capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed in {repo}: "
                           f"{r.stderr.decode('utf-8', 'replace')[:300]}")
    return r.stdout

workspace   E:\Thesis Project\Smell Ranker\workspace_data
repositories checkstyle, commons-lang, dubbo, junit-framework, questdb


## Section 1 — Inventory

### 1.1 Every `.java` file at every release

`git ls-tree` lists each release's tree. Releases are ordered by **ancestry** — how many commits
each can reach — rather than by the manifest or by date, because a maintenance release cut from
an old branch can be dated after a newer one.

In [14]:
def releases(repo):
    """(tag, sha) in ancestry order, oldest first."""
    entries = json.loads((VERSIONS / f"rel_hist_{repo}.json").read_text(encoding="utf-8"))
    rel = [(e.get("tag"), e.get("snapshot_sha")) for e in entries.get("entries", [])
           if e.get("snapshot_sha")]
    depth = {sha: int(git(repo, "rev-list", "--count", sha).decode().strip()) for _, sha in rel}
    return sorted(rel, key=lambda ts: depth[ts[1]])


def java_files(repo, sha):
    """(path, blob) for every .java file in the tree at `sha`. -z: no path quoting."""
    out = []
    for entry in git(repo, "ls-tree", "-r", "-z", sha).split(b"\0"):
        if not entry:
            continue
        meta, path = entry.split(b"\t", 1)
        path = path.decode("utf-8", "replace")
        if path.endswith(".java"):
            out.append((path, meta.split()[2].decode()))
    return out


RELEASES, FILES = {}, {}
for repo in REPOS:
    RELEASES[repo] = releases(repo)
    rows = []
    for pos, (tag, sha) in enumerate(RELEASES[repo]):
        for path, blob in java_files(repo, sha):
            rows.append((pos, tag, path, blob))
    FILES[repo] = pd.DataFrame(rows, columns=["pos", "release", "path", "blob"])
    f = FILES[repo]
    print(f"{repo:<18}{len(RELEASES[repo]):>4} releases   {f.path.nunique():>7,} distinct .java paths"
          f"   {len(f):>9,} (release, file) pairs")

checkstyle         204 releases    11,828 distinct .java paths     580,305 (release, file) pairs
commons-lang        35 releases     1,006 distinct .java paths      10,934 (release, file) pairs
dubbo              123 releases     8,068 distinct .java paths     287,752 (release, file) pairs
junit-framework     61 releases     2,410 distinct .java paths      79,623 (release, file) pairs
questdb            119 releases     9,263 distinct .java paths     330,057 (release, file) pairs


### 1.2 Each directory's source root, from its `package` declarations

One root per **directory**: every file in a directory belongs to the same package, so a few are
enough. Up to three files are read per directory and the most common answer is taken, because test
inputs sometimes declare a deliberately wrong package — that is what they test.

Only the first bytes of each file are needed, read once per distinct file version through
`git cat-file --batch`.

In [15]:
PACKAGE = re.compile(r"^\s*package\s+([A-Za-z_][\w.]*)\s*;", re.M)


def read_blobs(repo, blobs):
    """blob sha -> text, for many blobs in one git process."""
    raw = git(repo, "cat-file", "--batch", stdin="".join(b + "\n" for b in blobs).encode())
    out, i = {}, 0
    for b in blobs:
        nl = raw.index(b"\n", i)
        header = raw[i:nl].split()
        if len(header) < 3 or header[1] != b"blob":
            i = nl + 1
            continue
        size = int(header[2])
        out[b] = raw[nl + 1: nl + 1 + size].decode("utf-8", "replace")
        i = nl + 1 + size + 1
    return out


def root_of(directory, package):
    """
    The source root implied by a directory and the package its files declare, or None when the
    directory does not end in the package's path -- a file placed against Java's convention.
    """
    d = directory + "/" if directory else ""
    if package is None:                                  # default package: the directory is the root
        return d
    pkg = package.replace(".", "/") + "/"
    if d == pkg:
        return ""                                        # package tree starts at the repository root
    return d[: -len(pkg)] if d.endswith("/" + pkg) else None


def directory_roots(repo):
    f = FILES[repo].drop_duplicates("path")
    f = f.assign(dir=f.path.str.rpartition("/")[0])
    sample = f.groupby("dir").head(3)                    # up to three files per directory
    text = read_blobs(repo, sample.blob.unique().tolist())
    votes = collections.defaultdict(collections.Counter)
    for d, blob in zip(sample.dir, sample.blob):
        m = PACKAGE.search(text.get(blob, "")[:4000])
        votes[d][root_of(d, m.group(1) if m else None)] += 1
    roots = {}
    for d, c in votes.items():
        known = [(n, r) for r, n in c.items() if r is not None]
        roots[d] = max(known)[1] if known else None
    return roots


DIR_ROOT = {}
for repo in REPOS:
    DIR_ROOT[repo] = directory_roots(repo)
    f = FILES[repo]
    f["root"] = f.path.str.rpartition("/")[0].map(DIR_ROOT[repo])
    unknown = f[f.root.isna()].path.nunique()
    print(f"{repo:<18}{len(DIR_ROOT[repo]):>6,} directories   {f.root.nunique():>4} distinct roots"
          + (f"   {unknown:,} path(s) whose root could not be determined" if unknown else ""))

checkstyle         1,649 directories     97 distinct roots   105 path(s) whose root could not be determined
commons-lang          77 directories      5 distinct roots
dubbo              1,986 directories    325 distinct roots   4 path(s) whose root could not be determined
junit-framework      313 directories     95 distinct roots   1 path(s) whose root could not be determined
questdb              547 directories     16 distinct roots


### 1.3 The roots, with evidence

For each root: how many distinct files it held, when it first and last appeared, and two pieces of
evidence for the decision in section 2.

- **conventional** — `src/main/java/` or `src/java/`. Assumed to be production unless its path
  suggests otherwise.
- **hint** — a word in the path that usually marks non-production code: `test`, `example`,
  `benchmark`, `demo`, `doc` and similar, matched as a whole word, so `testkit` is not `test`.
- **shares classes with** — for a non-conventional root, how many of its classes (by package path
  and file name) also appear under a conventional root. A root sharing most of its classes with
  `src/main/java/` holds the same code, earlier or elsewhere.

In [16]:
HINTS = {"test", "tests", "it", "testing", "testinputs", "fixture", "fixtures", "example",
         "examples", "sample", "samples", "demo", "demos", "benchmark", "benchmarks", "jmh",
         "perf", "doc", "docs", "documentation", "xdocs", "tutorial", "tooling"}
CONVENTIONAL = re.compile(r"(?:^|/)src/(?:main/java|java)/$")


def hint_of(root):
    words = set(re.split(r"[/\-_.]", root.lower()))
    return ", ".join(sorted(words & HINTS))


def inventory(repo):
    f = FILES[repo].dropna(subset=["root"])
    tags = dict(enumerate(t for t, _ in RELEASES[repo]))
    inv = (f.groupby("root")
             .agg(files=("path", "nunique"), first=("pos", "min"), last=("pos", "max"))
             .reset_index())
    inv["conventional"] = inv.root.str.contains(CONVENTIONAL)
    inv["hint"] = inv.root.map(hint_of)

    # Classes each root held, by package path (the part below the root).
    # Files with no type are excluded: every root can hold a module-info.java, and matching it
    # would make unrelated modules look like the same code.
    typed = f.drop_duplicates("path")
    typed = typed[~typed.path.str.rsplit("/", n=1).str[-1].isin(NO_TYPE)]
    held = {r: set(g.path.str[len(r):]) for r, g in typed.groupby("root")}
    conv = [r for r in inv.root[inv.conventional]]
    shares, share_root = [], []
    for r, is_conv in zip(inv.root, inv.conventional):
        if is_conv or not conv:
            shares.append(None); share_root.append("")
            continue
        mine = held.get(r, set())
        best = max(conv, key=lambda c: len(mine & held.get(c, set())))
        n = len(mine & held.get(best, set()))
        shares.append(n / len(mine) if mine else 0.0)
        share_root.append(best if n else "")
    inv["shares"] = shares
    inv["shares_with"] = share_root
    inv["first_release"] = inv["first"].map(tags)
    inv["last_release"] = inv["last"].map(tags)
    return inv.sort_values(["conventional", "files"], ascending=[False, False])


INV = {}
for repo in REPOS:
    INV[repo] = inv = inventory(repo)
    print(f"\n{'=' * 100}\n{repo}   ({len(inv)} roots)\n{'=' * 100}")
    print(f"{'files':>7}  {'releases':<34}root   [evidence]")
    print("-" * 100)
    for _, r in inv.iterrows():
        notes = []
        if r.conventional:
            notes.append("conventional")
        elif r.shares and r.shares_with:
            notes.append(f"shares {r.shares * 100:.0f}% of its classes with {r.shares_with}")
        if r.hint:
            notes.append(f"hint: {r.hint}")
        span = f"{r.first_release} .. {r.last_release}"
        print(f"{r.files:>7,}  {span:<34}{r.root or '(repository root)'}"
              + (f"   [{'; '.join(notes)}]" if notes else ""))


checkstyle   (97 roots)
  files  releases                          root   [evidence]
----------------------------------------------------------------------------------------------------
    562  checkstyle-5.8 .. checkstyle-13.8.0src/main/java/   [conventional]
  4,626  checkstyle-5.8 .. checkstyle-13.8.0src/test/resources/   [hint: test]
  2,179  checkstyle-6.8 .. checkstyle-13.8.0src/it/resources/   [hint: it]
    910  checkstyle-10.12.1 .. checkstyle-13.8.0src/xdocs-examples/resources/   [hint: examples, xdocs]
    609  checkstyle-6.2 .. checkstyle-13.8.0src/test/resources-noncompilable/   [hint: test]
    567  checkstyle-5.8 .. checkstyle-13.8.0src/test/java/   [hint: test]
    529  checkstyle-6.8 .. checkstyle-13.8.0src/it/java/   [hint: it]
    493  release1_1 .. release5_7          src/checkstyle/   [shares 59% of its classes with src/main/java/]
    314  release1_1 .. release5_7          src/tests/   [hint: tests]
    214  checkstyle-10.12.1 .. checkstyle-13.8.0src/xdocs-examp

## Section 2 — Approved roots

**You decide here.** The next cell writes a proposal from section 1's evidence:

- conventional roots with no hint → **approved**
- conventional roots with a hint → listed but **commented out**, with the hint
- non-conventional roots sharing at least half their classes with a conventional root → listed
  but **commented out**, with the evidence — these are the historical roots

Copy it into `APPROVED` below, uncomment or remove lines, and run on. Nothing is approved until you
have done so: `APPROVED = None` stops the notebook.

The cell below holds the roots this study approved. To replicate the study, run it as it is.

In [17]:
SHARED_ENOUGH = 0.5

lines = ["APPROVED = {"]
for repo in REPOS:
    inv = INV[repo]
    lines.append(f'    "{repo}": [')
    for _, r in inv[inv.conventional].iterrows():
        if r.hint:
            lines.append(f'        # "{r.root}",   # hint: {r.hint}; {r.files:,} files')
        else:
            lines.append(f'        "{r.root}",   # {r.files:,} files, {r.first_release} .. {r.last_release}')
    cand = inv[(~inv.conventional) & (inv.shares.fillna(0) >= SHARED_ENOUGH)]
    for _, r in cand.iterrows():
        lines.append(f'        # "{r.root}",   # shares {r.shares * 100:.0f}% of its classes with '
                     f'{r.shares_with}; {r.first_release} .. {r.last_release}'
                     + (f'; hint: {r.hint}' if r.hint else ""))
    lines.append("    ],")
lines.append("}")
print("\n".join(lines))

APPROVED = {
    "checkstyle": [
        "src/main/java/",   # 562 files, checkstyle-5.8 .. checkstyle-13.8.0
        # "src/checkstyle/",   # shares 59% of its classes with src/main/java/; release1_1 .. release5_7
    ],
    "commons-lang": [
        "src/main/java/",   # 347 files, LANG_2_5 .. rel/commons-lang-3.20.0
        "src/java/",   # 83 files, LANG_1_0 .. LANG_2_4
    ],
    "dubbo": [
        "dubbo-common/src/main/java/",   # 683 files, dubbo-2.3.1 .. dubbo-3.3.6
        "dubbo-rpc/dubbo-rpc-triple/src/main/java/",   # 303 files, dubbo-3.0.0 .. dubbo-3.3.6
        "dubbo-cluster/src/main/java/",   # 302 files, dubbo-2.3.1 .. dubbo-3.3.6
        "dubbo-remoting/dubbo-remoting-api/src/main/java/",   # 249 files, dubbo-2.3.1 .. dubbo-3.3.6
        "dubbo-rpc/dubbo-rpc-api/src/main/java/",   # 197 files, dubbo-2.3.1 .. dubbo-3.3.6
        "dubbo-config/dubbo-config-spring/src/main/java/",   # 150 files, dubbo-2.3.1 .. dubbo-3.3.6
        "dubbo-registry/dubbo-registry-api/src/m

In [18]:
# The approved source roots. A .java file is production if its most specific source root is listed
# here; everything else is excluded. Reviewed against section 1's inventory.
APPROVED = {
    # Pre-migration root src/checkstyle/ (release1_1 .. release5_7) precedes src/main/java/ (5.8 ..).
    # Excluded: contrib/*, src/test*/, src/tests/, src/testinputs/, src/xdocs-examples/.
    "checkstyle": [
        "src/main/java/",
        "src/checkstyle/",
    ],

    # Ant layout src/java/ (LANG_1_0 .. LANG_2_4), then Maven src/main/java/ (LANG_2_5 ..).
    "commons-lang": [
        "src/main/java/",
        "src/java/",
    ],

    # Excluded: dubbo-demo/* (demos) and dubbo-test/* (test support, examples, benchmarks).
    # Kept: dubbo-admin, dubbo-simple/*, dubbo-maven-plugin and the compiler modules -- shipped
    # applications and tooling, hand-written, not demos or tests.
    "dubbo": [
        "dubbo-common/src/main/java/",
        "dubbo-rpc/dubbo-rpc-triple/src/main/java/",
        "dubbo-cluster/src/main/java/",
        "dubbo-remoting/dubbo-remoting-api/src/main/java/",
        "dubbo-rpc/dubbo-rpc-api/src/main/java/",
        "dubbo-config/dubbo-config-spring/src/main/java/",
        "dubbo-registry/dubbo-registry-api/src/main/java/",
        "dubbo-plugin/dubbo-qos/src/main/java/",
        "dubbo-config/dubbo-config-api/src/main/java/",
        "dubbo-admin/src/main/java/",
        "dubbo-rpc/dubbo-rpc-rest/src/main/java/",
        "dubbo-remoting/dubbo-remoting-http12/src/main/java/",
        "dubbo-metadata/dubbo-metadata-api/src/main/java/",
        "dubbo-compatible/src/main/java/",
        "dubbo-metrics/dubbo-metrics-api/src/main/java/",
        "hessian-lite/src/main/java/",
        "dubbo-rpc/dubbo-rpc-dubbo/src/main/java/",
        "dubbo-plugin/dubbo-rest-openapi/src/main/java/",
        "dubbo-xds/src/main/java/",
        "dubbo-native/src/main/java/",
        "dubbo-remoting/dubbo-remoting-netty4/src/main/java/",
        "dubbo-metadata/dubbo-metadata-processor/src/main/java/",
        "dubbo-metrics/dubbo-metrics-default/src/main/java/",
        "dubbo-remoting/dubbo-remoting-http/src/main/java/",
        "dubbo-remoting/dubbo-remoting-p2p/src/main/java/",
        "dubbo-plugin/dubbo-rest-jaxrs/src/main/java/",
        "dubbo-metrics/dubbo-tracing/src/main/java/",
        "dubbo-maven-plugin/src/main/java/",
        "dubbo-plugin/dubbo-native/src/main/java/",
        "dubbo-rpc/dubbo-rpc-default/src/main/java/",
        "dubbo-rpc/dubbo-rpc-thrift/src/main/java/",
        "dubbo-filter/dubbo-filter-cache/src/main/java/",
        "dubbo-registry/dubbo-registry-xds/src/main/java/",
        "dubbo-remoting/dubbo-remoting-zookeeper/src/main/java/",
        "dubbo-serialization/dubbo-serialization-native-hession/src/main/java/",
        "dubbo-serialization/dubbo-serialization-api/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-autoconfigure/src/main/java/",
        "dubbo-plugin/dubbo-rest-spring/src/main/java/",
        "dubbo-serialization/dubbo-serialization-hessian2/src/main/java/",
        "dubbo-serialization/dubbo-serialization-kryo/src/main/java/",
        "dubbo-remoting/dubbo-remoting-netty/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-autoconfigure/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-actuator/src/main/java/",
        "dubbo-monitor/dubbo-monitor-api/src/main/java/",
        "dubbo-serialization/dubbo-serialization-jdk/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-compatible/actuator/src/main/java/",
        "dubbo-filter/dubbo-filter-validation/src/main/java/",
        "dubbo-remoting/dubbo-remoting-etcd3/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-observability-starters/autoconfigure/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-starters/observability/autoconfigure/src/main/java/",
        "dubbo-plugin/dubbo-filter-cache/src/main/java/",
        "dubbo-registry/dubbo-registry-nacos/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-compatible/autoconfigure/src/main/java/",
        "dubbo-container/dubbo-container-api/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-definition/src/main/java/",
        "dubbo-remoting/dubbo-remoting-http3/src/main/java/",
        "dubbo-rpc/dubbo-rpc-hessian/src/main/java/",
        "dubbo-plugin/dubbo-auth/src/main/java/",
        "dubbo-plugin/dubbo-triple-servlet/src/main/java/",
        "dubbo-simple/dubbo-monitor-simple/src/main/java/",
        "dubbo-plugin/dubbo-mutiny/src/main/java/",
        "dubbo-plugin/dubbo-reactive/src/main/java/",
        "dubbo-remoting/dubbo-remoting-grizzly/src/main/java/",
        "dubbo-remoting/dubbo-remoting-mina/src/main/java/",
        "dubbo-rpc/dubbo-rpc-grpc/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-actuator/src/main/java/",
        "dubbo-metrics/dubbo-metrics-event/src/main/java/",
        "dubbo-plugin/dubbo-spring6-security/src/main/java/",
        "dubbo-registry/dubbo-registry-zookeeper/src/main/java/",
        "dubbo-remoting/dubbo-remoting-websocket/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot/src/main/java/",
        "dubbo-kubernetes/src/main/java/",
        "dubbo-metrics/dubbo-metrics-registry/src/main/java/",
        "dubbo-plugin/dubbo-filter-validation/src/main/java/",
        "dubbo-remoting/dubbo-remoting-zookeeper-curator5/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-api/src/main/java/",
        "dubbo-monitor/dubbo-monitor-default/src/main/java/",
        "dubbo-registry/dubbo-registry-kubernetes/src/main/java/",
        "dubbo-rpc/dubbo-rpc-http/src/main/java/",
        "dubbo-rpc/dubbo-rpc-injvm/src/main/java/",
        "dubbo-compiler/src/main/java/",                      # code generator -- see note above
        "dubbo-metrics/dubbo-metrics-metadata/src/main/java/",
        "dubbo-plugin/dubbo-spring-security/src/main/java/",
        "dubbo-serialization/dubbo-serialization-fst/src/main/java/",
        "dubbo-serialization/dubbo-serialization-protobuf/src/main/java/",
        "dubbo-serialization/dubbo-serialization-protostuff/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-observability-starter/src/main/java/",
        "compiler/src/main/java/",                            # code generator -- see note above
        "dubbo-config/dubbo-config-spring6/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-api/src/main/java/",
        "dubbo-plugin/dubbo-security/src/main/java/",
        "dubbo-registry/dubbo-registry-dns/src/main/java/",
        "dubbo-plugin/dubbo-compiler/src/main/java/",         # code generator -- see note above
        "dubbo-plugin/dubbo-triple-websocket/src/main/java/",
        "dubbo-registry/dubbo-registry-consul/src/main/java/",
        "dubbo-registry/dubbo-registry-multicast/src/main/java/",
        "dubbo-registry/dubbo-registry-multiple/src/main/java/",
        "dubbo-registry/dubbo-registry-redis/src/main/java/",
        "dubbo-registry/dubbo-registry-sofa/src/main/java/",
        "dubbo-serialization/dubbo-serialization-fastjson/src/main/java/",
        "dubbo-serialization/dubbo-serialization-fastjson2/src/main/java/",
        "dubbo-serialization/dubbo-serialization-protobuf-json/src/main/java/",
        "dubbo-plugin/dubbo-qos-api/src/main/java/",
        "dubbo-remoting/dubbo-remoting-redis/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-actuator-autoconfigure/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-zookeeper/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-failover/src/main/java/",
        "dubbo-metrics/dubbo-metrics-config-center/src/main/java/",
        "dubbo-metrics/dubbo-metrics-netty/src/main/java/",
        "dubbo-metrics/dubbo-metrics-prometheus/src/main/java/",
        "dubbo-native-plugin/src/main/java/",
        "dubbo-registry/dubbo-registry-default/src/main/java/",
        "dubbo-rpc/dubbo-rpc-rmi/src/main/java/",
        "dubbo-serialization/dubbo-serialization-gson/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-compatible/dubbo-spring-boot-actuator-autoconfigure-compatible/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-nacos/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-nacos/src/main/java/",
        "dubbo-registry/dubbo-registry-etcd3/src/main/java/",
        "dubbo-registry/dubbo-registry-eureka/src/main/java/",
        "dubbo-rpc/dubbo-rpc-http-invoker/src/main/java/",
        "dubbo-serialization/dubbo-serialization-avro/src/main/java/",
        "dubbo-serialization/dubbo-serialization-msgpack/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-apollo/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-consul/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-etcd/src/main/java/",
        "dubbo-configcenter/dubbo-configcenter-file/src/main/java/",
        "dubbo-container/dubbo-container-log4j/src/main/java/",
        "dubbo-container/dubbo-container-logback/src/main/java/",
        "dubbo-container/dubbo-container-spring/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-consul/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-etcd/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-nacos/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-redis/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-report-zookeeper/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-consul/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-etcd/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-redis/src/main/java/",
        "dubbo-metadata/dubbo-metadata-report-zookeeper/src/main/java/",
        "dubbo-rpc/dubbo-rpc-memcached/src/main/java/",
        "dubbo-rpc/dubbo-rpc-redis/src/main/java/",
        "dubbo-rpc/dubbo-rpc-webservice/src/main/java/",
        "dubbo-rpc/dubbo-rpc-xml/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-3-autoconfigure/src/main/java/",
        "dubbo-spring-boot-project/dubbo-spring-boot-compatible/dubbo-spring-boot-autoconfigure-compatible/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-3-autoconfigure/src/main/java/",
        "dubbo-spring-boot/dubbo-spring-boot-interceptor/src/main/java/",
        "dubbo-bootstrap/src/main/java/",
        "dubbo-container/dubbo-container-jetty/src/main/java/",
        "dubbo-metadata-report/dubbo-metadata-definition-protobuf/src/main/java/",
        "dubbo-metadata/dubbo-metadata-definition-protobuf/src/main/java/",
        "dubbo-plugin/dubbo-plugin-loom/src/main/java/",
        "dubbo-rpc/dubbo-rpc-jsonrpc/src/main/java/",
        "dubbo-rpc/dubbo-rpc-native-thrift/src/main/java/",
        "dubbo-simple/dubbo-registry-simple/src/main/java/",
    ],

    # Excluded: documentation/ and platform-tooling-support-tests/* (fixture projects);
    # src/main/java9/ and src/main/java17/ (multi-release variants of classes already in
    # src/main/java/ -- approving them would give one class two files); src/module/* (only
    # module-info.java, which holds no class).
    "junit-framework": [
        "junit-jupiter-api/src/main/java/",
        "junit-jupiter-engine/src/main/java/",
        "junit-platform-engine/src/main/java/",
        "junit-platform-launcher/src/main/java/",
        "junit-jupiter-params/src/main/java/",
        "junit-platform-console/src/main/java/",
        "junit-platform-commons/src/main/java/",
        "junit-vintage-engine/src/main/java/",
        "junit-platform-suite-api/src/main/java/",
        "junit-jupiter-migrationsupport/src/main/java/",
        "junit-platform-testkit/src/main/java/",
        "junit-platform-reporting/src/main/java/",
        "junit-platform-suite-engine/src/main/java/",
        "junit-platform-jfr/src/main/java/",
        "junit-platform-surefire-provider/src/main/java/",
        "junit-platform-runner/src/main/java/",
        "junit-platform-suite-commons/src/main/java/",
        "junit-start/src/main/java/",
        "junit-jupiter/src/main/java/",
        "junit-platform-commons-java-9/src/main/java/",
        "junit-platform-suite/src/main/java/",
    ],

    # NFSdb was questdb's name before 3.0: nfsdb-core (1.0 .. 2.1.0) precedes core (3.0.0 ..).
    # Excluded: benchmarks/, examples/, nfsdb-examples/, nfsdb-samples/, nfsdb-test-model/.
    "questdb": [
        "core/src/main/java/",
        "nfsdb-core/src/main/java/",
        "utils/src/main/java/",
        "nfsdb-guice/src/main/java/",
        "nfsdb-thrift/src/main/java/",
    ],
}

if APPROVED is None:
    raise SystemExit("APPROVED is not set. Paste the proposal from the cell above, review each "
                     "line, and run again.")
missing = [r for r in REPOS if not APPROVED.get(r)]
if missing:
    raise SystemExit(f"no approved root for: {', '.join(missing)}")
duplicates = {r: [a for a in set(v) if v.count(a) > 1] for r, v in APPROVED.items()}
duplicates = {r: d for r, d in duplicates.items() if d}
if duplicates:
    raise SystemExit(f"listed twice: {duplicates}")
print("approved roots: " + "   ".join(f"{r} {len(v)}" for r, v in APPROVED.items()))

approved roots: checkstyle 2   commons-lang 2   dubbo 154   junit-framework 21   questdb 5


In [19]:
# The definition, written once and read by everything downstream: Notebooks 1 to 3.
# ALL known roots are recorded, not only the
# approved ones, so a path is classified by its MOST SPECIFIC root -- src/test/java/ must win over
# a shorter approved root such as src/ if a repository ever has one.
SCOPE = {}
for repo in REPOS:
    inv = INV[repo]
    SCOPE[repo] = {
        "approved": sorted(APPROVED[repo]),
        "roots": {r.root: {"files": int(r.files), "conventional": bool(r.conventional),
                           "first": r.first_release, "last": r.last_release,
                           "approved": r.root in APPROVED[repo]}
                  for _, r in inv.iterrows()},
    }
    unknown = [a for a in APPROVED[repo] if a not in SCOPE[repo]["roots"]]
    if unknown:
        raise SystemExit(f"{repo}: approved root(s) not found in the inventory: {unknown}")

path = OUTPUTS / "production_scope.json"
path.write_text(json.dumps({"rule": "a .java file is production if its most specific known "
                                    "source root is approved; package-info.java and "
                                    "module-info.java hold no type and are excluded",
                            "repositories": SCOPE}, indent=1), encoding="utf-8")
print(f"wrote {path}")
for repo in REPOS:
    print(f"   {repo:<18}{', '.join(SCOPE[repo]['approved'])}")

wrote E:\Thesis Project\Smell Ranker\workspace_data\outputs\production_scope.json
   checkstyle        src/checkstyle/, src/main/java/
   commons-lang      src/java/, src/main/java/
   dubbo             compiler/src/main/java/, dubbo-admin/src/main/java/, dubbo-bootstrap/src/main/java/, dubbo-cluster/src/main/java/, dubbo-common/src/main/java/, dubbo-compatible/src/main/java/, dubbo-compiler/src/main/java/, dubbo-config/dubbo-config-api/src/main/java/, dubbo-config/dubbo-config-spring/src/main/java/, dubbo-config/dubbo-config-spring6/src/main/java/, dubbo-configcenter/dubbo-configcenter-api/src/main/java/, dubbo-configcenter/dubbo-configcenter-apollo/src/main/java/, dubbo-configcenter/dubbo-configcenter-consul/src/main/java/, dubbo-configcenter/dubbo-configcenter-etcd/src/main/java/, dubbo-configcenter/dubbo-configcenter-file/src/main/java/, dubbo-configcenter/dubbo-configcenter-nacos/src/main/java/, dubbo-configcenter/dubbo-configcenter-zookeeper/src/main/java/, dubbo-container/dubbo-

## Section 3 — Classify every file, and check continuity

Every `(release, path)` is classified by the rule written above — the **same function** the rest of
the pipeline will use, applied here to every file git knows about.

**Continuity is the completeness check.** Production code grows and shrinks gradually. A release
with **no** production files, or with far fewer than the one before, almost always means a root in
use at that time has not been approved. The table flags both.

In [20]:
def classifier(scope):
    """path -> (category, root). Most specific known root wins."""
    roots = sorted(scope["roots"], key=len, reverse=True)
    approved = set(scope["approved"])

    def classify(path):
        root = next((r for r in roots if path.startswith(r)), None)
        if root is None:
            return "root unknown", None
        if root not in approved:
            return "not approved", root
        if path.rsplit("/", 1)[-1] in NO_TYPE:
            return "no type", root
        return "production", root
    return classify


DROP, SHARP = 0.5, []
for repo in REPOS:
    classify = classifier(SCOPE[repo])
    f = FILES[repo]
    cats = {p: classify(p) for p in f.path.unique()}
    f["category"] = f.path.map(lambda p: cats[p][0])
    f["rule_root"] = f.path.map(lambda p: cats[p][1])

    per = (f[f.category == "production"].groupby("pos").size()
             .reindex(range(len(RELEASES[repo])), fill_value=0))
    flagged = []
    for pos, n in per.items():
        prev = per.get(pos - 1)
        if n == 0:
            flagged.append((pos, n, "no production files"))
        elif prev and n < DROP * prev:
            flagged.append((pos, n, f"fell from {prev:,}"))

    print(f"\n{repo}")
    print(f"   production files per release: first {per.iloc[0]:,}   median {int(per.median()):,}"
          f"   max {per.max():,}   last {per.iloc[-1]:,}")
    cat = f.drop_duplicates("path").category.value_counts()
    print("   distinct paths: " + "   ".join(f"{k} {v:,}" for k, v in cat.items()))
    if flagged:
        print(f"   {len(flagged)} release(s) flagged:")
        for pos, n, why in flagged[:12]:
            print(f"      {RELEASES[repo][pos][0]:<28}{n:>6,}   {why}")
        SHARP.append(repo)
    else:
        print("   continuous: no release empty or collapsed")

print("\n" + ("Flagged repositories: " + ", ".join(SHARP) +
              ". Check section 1 for a root those releases used, approve it, and rerun from "
              "section 2." if SHARP else "Every repository is continuous across its releases."))


checkstyle
   production files per release: first 13   median 356   max 449   last 449
   distinct paths: not approved 10,773   production 1,001   no type 54
   continuous: no release empty or collapsed

commons-lang
   production files per release: first 26   median 121   max 241   last 241
   distinct paths: not approved 576   production 412   no type 18
   continuous: no release empty or collapsed

dubbo
   production files per release: first 513   median 1,387   max 2,370   last 2,370
   distinct paths: production 4,721   not approved 3,339   root unknown 4   no type 4
   continuous: no release empty or collapsed

junit-framework
   production files per release: first 330   median 621   max 791   last 791
   distinct paths: not approved 1,378   production 939   no type 92   root unknown 1
   continuous: no release empty or collapsed

questdb
   production files per release: first 120   median 1,890   max 3,489   last 3,489
   distinct paths: production 5,720   not approved 3,540  

In [21]:
summary = []
for repo in REPOS:
    f = FILES[repo]
    out = OUTPUTS / f"production_files_{repo}.csv"
    f[["pos", "release", "path", "category", "rule_root"]].rename(
        columns={"pos": "release_pos", "rule_root": "root"}).to_csv(out, index=False)
    prod = f[f.category == "production"]
    summary.append({"repo": repo, "releases": len(RELEASES[repo]),
                    "production_paths": prod.path.nunique(),
                    "production_file_releases": len(prod),
                    "excluded_paths": f[f.category != "production"].path.nunique(),
                    "approved_roots": len(SCOPE[repo]["approved"]),
                    "continuous": repo not in SHARP})
    print(f"wrote {out.name}   ({len(f):,} rows)")

S = pd.DataFrame(summary)
S.to_csv(OUTPUTS / "production_summary.csv", index=False)
print(f"wrote production_summary.csv\n")
print(S.to_string(index=False))

wrote production_files_checkstyle.csv   (580,305 rows)
wrote production_files_commons-lang.csv   (10,934 rows)
wrote production_files_dubbo.csv   (287,752 rows)
wrote production_files_junit-framework.csv   (79,623 rows)
wrote production_files_questdb.csv   (330,057 rows)
wrote production_summary.csv

           repo  releases  production_paths  production_file_releases  excluded_paths  approved_roots  continuous
     checkstyle       204              1001                     71187           10827               2        True
   commons-lang        35               412                      4816             594               2        True
          dubbo       123              4721                    160441            3347             154        True
junit-framework        61               939                     36122            1471              21        True
        questdb       119              5720                    230469            3543               5        True


## Section 4 — Which refactored files are production?

RefactoringMiner reports every `.java` file a refactoring touched, test code included. This section
classifies those files by the same rule, so it is clear how much refactoring activity the study
leaves out, and why.

Counted in two units:

- **distinct files** — every file any refactoring touched, over the whole history
- **(interval, file) pairs** — the unit Notebook 1 labels: a file refactored at least once in an
  interval. Move Source Folder is left out, as in Notebook 1.

Non-production files are split in two, because Aniche et al. exclude only the first:

- **test** — a root whose path contains *test, tests, it, testing, testinputs* or *fixture(s)*.
  Aniche et al.: *"our approach ignores test code (e.g., JUnit files) and only captures refactoring
  operations in production files."*
- **other non-production** — examples, demos, benchmarks, documentation, contributed modules. Excluded
  here, but not by Aniche et al.

In [31]:
if "SCOPE" not in globals():
    raise SystemExit("run sections 1-3 first: section 4 classifies with their rule")

TEST_WORDS = {"test", "tests", "it", "testing", "testinputs", "fixture", "fixtures"}
NOT_A_REFACTORING = {"move source folder"}              # as in Notebook 1


def find_output(repo, name):
    for d in (OUTPUTS / repo, OUTPUTS):
        if (d / name).exists():
            return d / name
    return None


def kind_of(category, root):
    """production / test / other non-production / no type / root unknown."""
    if category == "production":
        return "production"
    if category in ("no type", "root unknown"):
        return category
    words = set(re.split(r"[/\-_.]", (root or "").lower()))
    return "test" if words & TEST_WORDS else "other non-production"


KINDS = ["production", "test", "other non-production", "no type", "root unknown"]
REFM_SUMMARY = []
for repo in REPOS:
    refm = find_output(repo, f"refactorings_{repo}.jsonl")
    ivf = find_output(repo, f"intervals_{repo}.json")
    if refm is None or ivf is None:
        print(f"{repo}: refactorings or intervals file not found -- skipped")
        continue
    iv = json.loads(ivf.read_text(encoding="utf-8"))
    position = {x["interval"]: i for i, x in enumerate(iv["interval_index"])}
    interval_of = {sha: position[name] for sha, name in iv["map"].items()}
    classify = classifier(SCOPE[repo])

    commits = collections.Counter()
    files_all, pairs, roots = {}, set(), collections.Counter()
    for rec in (json.loads(l) for l in open(refm, encoding="utf-8") if l.strip()):
        if not rec.get("refactorings"):
            continue
        i = interval_of.get(rec["sha1"])
        commits["inside an interval" if i is not None else "outside every interval"] += 1
        for r in rec["refactorings"]:
            if str(r.get("type", "")).strip().lower() in NOT_A_REFACTORING:
                continue
            for loc in r.get("leftSideLocations") or []:
                path = str(loc.get("filePath", "")).replace("\\", "/")
                if not path.endswith(".java"):
                    continue
                if path not in files_all:
                    cat, root = classify(path)
                    files_all[path] = (kind_of(cat, root), root)
                if i is not None:
                    pairs.add((i, path))

    by_file = collections.Counter(k for k, _ in files_all.values())
    by_pair = collections.Counter(files_all[p][0] for _, p in pairs)
    for _, p in pairs:
        k, root = files_all[p]
        if k != "production":
            roots[(k, root or "(no known root)")] += 1

    n_files, n_pairs = len(files_all), len(pairs)
    print(f"\n{'=' * 90}\n{repo}\n{'=' * 90}")
    print(f"   refactoring commits: " + "   ".join(f"{k} {v:,}" for k, v in commits.items()))
    print(f"\n   {'':<24}{'distinct files':>16}{'(interval, file) pairs':>26}")
    for k in KINDS:
        if by_file.get(k) or by_pair.get(k):
            print(f"   {k:<24}{by_file.get(k, 0):>9,} ({by_file.get(k, 0) / max(1, n_files):>5.1%})"
                  f"{by_pair.get(k, 0):>15,} ({by_pair.get(k, 0) / max(1, n_pairs):>5.1%})")
    print(f"   {'total':<24}{n_files:>9,}{'':>9}{n_pairs:>15,}")

    aniche = by_pair.get("production", 0) + by_pair.get("other non-production", 0)
    print(f"\n   refactored pairs kept   this study {by_pair.get('production', 0) / max(1, n_pairs):.1%}"
          f"   Aniche et al.'s rule (tests out) {aniche / max(1, n_pairs):.1%}")

    if roots:
        print("\n   where the non-production refactorings are (pairs, top roots):")
        for (k, root), n in roots.most_common(8):
            print(f"      {n:>7,}  {k:<22}{root}")


    REFM_SUMMARY.append({"repo": repo, "refactoring_commits": sum(commits.values()),
                         "in_intervals": commits.get("inside an interval", 0),
                         "refactored_files": n_files, "refactored_pairs": n_pairs,
                         **{f"pairs_{k}": by_pair.get(k, 0) for k in KINDS},
                         "kept_this_study": round(by_pair.get("production", 0) / max(1, n_pairs), 4),
                         "kept_aniche_rule": round(aniche / max(1, n_pairs), 4)})
    pd.DataFrame([{"kind": k, "root": r, "pairs": n} for (k, r), n in roots.most_common()]) \
        .to_csv(OUTPUTS / f"refactored_roots_{repo}.csv", index=False)

R5 = pd.DataFrame(REFM_SUMMARY)
R5.to_csv(OUTPUTS / "refactored_scope_summary.csv", index=False)
print(f"\n\nwrote refactored_scope_summary.csv and refactored_roots_<repo>.csv\n")
print(R5[["repo", "refactoring_commits", "refactored_pairs", "pairs_production", "pairs_test",
          "pairs_other non-production", "kept_this_study", "kept_aniche_rule"]].to_string(index=False))


checkstyle
   refactoring commits: inside an interval 3,792   outside every interval 36

                             distinct files    (interval, file) pairs
   production                    852 (14.7%)          3,840 (33.7%)
   test                        4,467 (76.8%)          6,824 (59.8%)
   other non-production          494 ( 8.5%)            741 ( 6.5%)
   root unknown                    1 ( 0.0%)              1 ( 0.0%)
   total                       5,814                  11,406

   refactored pairs kept   this study 33.7%   Aniche et al.'s rule (tests out) 40.2%

   where the non-production refactorings are (pairs, top roots):
        2,134  test                  src/test/java/
        1,911  test                  src/test/resources/
        1,251  test                  src/it/resources/
          835  test                  src/it/java/
          441  other non-production  src/xdocs-examples/java/
          347  test                  src/tests/
          269  other non-produc